In [1]:
import argparse
import os
import pickle

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "2.2.4":
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:

from kawada_env_cnoid import KawadaBaseEnvChoreonoid as RL_Env

In [3]:

# from bex24_env_cnoid import RLEnvChoreonoid as RL_Env

In [4]:
# exp_name = 'kawada-walking-1001'
# ckpt = 1000

In [5]:
exp_name = 'ishiki-walking-rand'
ckpt = 1000

In [6]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}
env_cfg["rotorInertia"] = 0.3

In [7]:
env = RL_Env(
        num_envs=1,
        env_cfg=env_cfg,
        obs_cfg=obs_cfg,
        reward_cfg=reward_cfg,
        command_cfg=command_cfg,
        dt=env_cfg['dt'],
        substeps=env_cfg['substeps'],
        show_viewer=True,
    )

In [8]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

Actor MLP: Sequential(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: Sequential(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)


/usr/local/lib/python3.10/dist-packages/_distutils_hack/__init__.py:53: UserWarning: Reliance on distutils from stdlib is deprecated. Users must rely on setuptools to provide the distutils module. Avoid importing distutils or import setuptools first, and avoid setting SETUPTOOLS_USE_DISTUTILS=stdlib. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


In [9]:
with torch.no_grad():
        print("cnt :", cnt)   
        actions = policy(obs)
        print("2 : ", actions)
        obs, rews, dones, infos = env.step(actions)
        print(obs)
        torques = env.sim.sbody.getTorques()
        print("torques:", torques)
        cnt += 1

cnt : 0
2 :  tensor([[ 0.0260,  0.1785, -1.1930, -0.1154, -2.6597,  1.6658, -0.1468,  1.1028,
         -1.5938, -0.4599, -3.1214,  0.6065]], device='cuda:0')


/userdir/samples/../irsl_rl/rl_env_base.py:96: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/samples/../irsl_rl/rl_env_cnoid.py:84: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  self.dof_pos = torch.tensor([sbody.angleVector()]).to(torch.float32).to(self.device)


tensor([[-5.1573e-07, -1.2704e-02,  1.5645e-06,  4.6624e-11,  2.1482e-11,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00,  2.9924e-08,
          2.6430e-09, -2.0790e-04,  6.8772e-04, -3.7783e-04, -3.3605e-08,
          1.9338e-07, -2.9767e-08, -2.0766e-04,  6.8736e-04, -3.7855e-04,
          4.0762e-08,  7.4798e-07,  6.4796e-08, -5.1980e-03,  1.7193e-02,
         -9.4461e-03, -8.4016e-07,  4.8349e-06, -7.4548e-07, -5.1915e-03,
          1.7185e-02, -9.4641e-03,  1.0190e-06,  2.6038e-02,  1.7851e-01,
         -1.1930e+00, -1.1543e-01, -2.6597e+00,  1.6658e+00, -1.4676e-01,
          1.1028e+00, -1.5938e+00, -4.5989e-01, -3.1214e+00,  6.0653e-01]],
       device='cuda:0')
torques: [-1.23943213e-06 -2.19489343e-05  7.63267114e-05 -1.03964466e-04
  1.08139051e-03 -6.22839858e-07  4.58367113e-06 -2.22571853e-05
  3.21343125e-05  3.37616597e-05 -6.93686110e-04 -5.70733356e-07]


In [10]:
for i in range(50):
    with torch.no_grad():
        actions = policy(obs)
        print("2 : ", actions)
        torques = env.sim.sbody.getTorques()
        print("torquse:",torques)
        obs, rews, dones, infos = env.step(actions)

torquse: [-1.23943213e-06 -2.19489343e-05  7.63267114e-05 -1.03964466e-04
  1.08139051e-03 -6.22839858e-07  4.58367113e-06 -2.22571853e-05
  3.21343125e-05  3.37616597e-05 -6.93686110e-04 -5.70733356e-07]
torquse: [  13.33034429   87.63237017 -500.          -71.23488306 -500.
  500.          -74.64983785  500.         -500.         -230.02125029
 -500.          322.6305603 ]
torquse: [-500.         -310.94295823  291.11212629  129.53544384 -500.
  500.         -497.22000677 -462.10701553  253.697626    489.88470155
  500.         -436.54408175]
torquse: [ 402.99878686 -296.27126226 -180.07570679    9.5442081  -500.
    8.49146257  151.58112019  500.         -500.         -374.81519829
  500.          500.        ]
torquse: [-500.          175.80541509  500.         -500.          500.
  500.         -500.         -500.         -213.75991312 -132.2488627
 -500.         -451.14069543]
torquse: [-500.         -271.94603952  -31.5411468   -85.61908692  500.
  -37.28035292 -500.         -15

In [11]:
# for i in range(500):
#     obs, _ = env.reset()
#     with torch.no_grad():
#         actions = policy(obs)
#         obs, rews, dones, infos = env.step(actions)

In [12]:
env.sim.stop()